---
title: Best Practice Guide - Python front-ends for HPC
author:
  - name: Engelbert Tijskens
    corresponding: true
    email: engelbert.tijskens@uantwerpen.be
    affiliations:
      - Universiteit Antwerpen
      - Vlaams Supercomputer Centrum
keywords:
  - HPC
  - Python
  - binary extension modules
  - Development of research applications 
abstract: |
  Python front-ends combine the expressiveness and flexibility of an interpreted general purpose high-level programming and scripting language with the performance of compiled low-level languaged such as C++ and Fortran. This is especially usefull in High Performance Programming. This can be considered the core of a strategy for developing research software for small teams. 
  
  We acknowledge **EPICURE**, a EuroHPC Joint Undertaking initiative for supporting this project.
key-points:
  - why Python and performance do not always play well together and what you can do about that
  - A practical way of building applications for scientific computing
  - tools for building binary Python extension modules
date: last-modified
bibliography: references.bib
number-sections: true
---

## Introduction

### Why Python?

The [yearly Stack Overflow survey of 2025](https://survey.stackoverflow.co/2025/technology#1-programming-scripting-and-markup-languages) tells us that Python is the 4th most popular programming/scripting/mark up language and saw a 7 % increase in popularity w.r.t. 2024. In fact, it is the most popular *general purpose* programming/scripting language.

Here are some features that have contributed to its general popularity:

- Readibility: Python is quick and easy it is to write and understand. Its syntax is simple and closely resembles natural language, making it especially beginner-friendly. Python docstrings encourages documenting your code, by making it available in interactive environments and IDEs. In addition, Python is quite an elegant and expressive language. As beauty, in general, is very appealing to humans, so is a well written Python script. 
- Batteries included: Python comes with an impressive Standard Library
- `pip install whatever`: An even larger suite of third party libraries is available from your finger tips in the [Python Package Index](https://pypi.org)
- Portability across hardware and operating systems
- Python, being an interpreted language, allows for interactive use, and makes debugging and testing practical and easy
- Tooling: a rich set of tools is available that help with managing your Python installations, virtual environments, testing, profiling, continuous integration, notebooks, ...
- Because Python is free and open source, there is a huge and strong community, code and documentation is voluntarily shared

In general, your productivity as a programmer is likely to increase a lot by using Python instead of compiled languages. 

However, scientific computing is a niche where performance is often of crucial importance, and for HPC this is even more so. It is well know that dynamical typing and being an interpreted language comes at a runtime cost, and it is not difficult to write Python code that can be outperformed by compiled C++ or Fortran by several orders of magnitude. Still, also for scientific computing Python is very popular.

- Many scientific computation packages come with a Python API. Some examples:
    - [ASE, Atomic Simulation Environment](https://ase-lib.org), for orchestrating atomistic simulations with NWChem, Gromacs, LAMMPs, Quantum Espresso, Siesta, and many many more
    - pyELPA: a Python interface to [ELPA](https://elpa.mpcdf.mpg.de), a library providing **E**igenvalue so**L**vers for **P**etaflop **A**pplications.

  These two examples present two ends of a spectrum. ELPA is a shared library written in C++ which can be linked into other applications written in C++, Fortran, ... that need to solve large eigenvalue problems. It comes with a Python interface, pyELPA, that can be imported into a Python script, but the performance critical code is written in a low level language, _c.q._ C++. On the other end of the spectrum is ASE that provides a generic approach of setting up atomistic simulations in a Python script. ASE is able to translate this generic input for the chosen `calculator` objects, which can be selected from a wide range of atomistic simulation applications. ASE does 'only' provide wrappers for these applications, it does not have any low-level performance critical code of its own.

- [Numpy](https://numpy.org): a Python package for creating and manipulation arrays, with a very strong focus on scientific computing
    - Powerful N-dimensional arrays: Fast and versatile, the NumPy vectorization, indexing, and broadcasting concepts are the de-facto standards of array computing today.
    - Numerical computing tools: NumPy offers comprehensive mathematical functions, random number generators, linear algebra routines, Fourier transforms, and more.
    - Interoperable: NumPy supports a wide range of hardware and computing platforms, and plays well with distributed, GPU, and sparse array libraries.
    - Performant: The core of NumPy is well-optimized C code. Enjoy the flexibility of Python with the speed of compiled code.

    That last line is important, because this the feature that gives us all the advantages of Python, yet the maximum of performance. The Python module provides a high level interface to a scientific computing object, _c.q._ multi-dimensional arrays, but the hard number-crunching work is done in a highly optimized library compiled from C. In fact, as will be demonstrated in this _Best Practice Guide_, building your own Python packages that run compiled low-level code under the hood isn't hard at all. It should come to no surprise that scientific computing (in Python) focuses on (Numpy) arrays. Hardware (CPUs as well as GPUs) is built for maximum performance when addressing data (as well as instructions) in a contigous manner, that is, for traversing arrays linearly. 

    > **Quote**
    > 
    >"I don't know what your fancy data structure is, but I know an array is gonna beat it" [Scott meyers in "_CPU Caches and Why You Care_", slide 34, 45'20"](https://www.youtube.com/watch?v=WDIkqP4JbkE) (Despite the fact that it dates back from 2014, still extremely relevant). The talk explains extremely well why arrays are the fastest data structure and how they are preferentially used. For GPUs check this talk by Stephen Jones from NVIDIA: [How GPU computing works](https://www.nvidia.com/en-us/on-demand/session/gtcspring21-s31151/)
  


- Numpy has become the _de facto_ standard for scientific computing with arrays in Python. Many scientific computing packages depend on it. Some examples (far from exhaustive):
    - [Jax](https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html) array-oriented numerical computation (à la NumPy), with [automatic differentiation](https://en.wikipedia.org/wiki/Automatic_differentiation) and JIT compilation
    - [SciPy](https://scipy.org) Fundamental algorithms for scientific computing in Python
    - [Numba](numba.pydata.org) open source JIT compiler that translates a subset of Python and NumPy code into fast machine code
    - [sympy](https://www.sympy.org/en/index.html]) symbolic computation
    - [pyTorch](https://pyTorch.org) Machine learning
    - [pandas](https://pandas.pydata.org) and [polars](https://pola.rs) data science
    - [mpi4py](mpi4py.readthedocs.io), [dask](https://www.dask.org) for parallel computing (multi-core and multi-node)
    - [h5py](https://docs.h5py.org/en/stable/) reading and writing hdf5 files in Python, including parallel I/O
    - [matplotlib](https://matplotlib.org), [seaborn](https://seaborn.pydata.org), [plotly](https://plotly.com) postprocessing and visualization
    - ...

  These are all high-quality, well-documented, highly optimized packages.

### A strategy for developing research software

As many scientific computing problems can be expressed in terms of (Numpy) arrays and functionality already present in aforementioned Python packages, and given the many advantages of Python, the following strategy for developing research software in Python is proposed. As the central idea is to write as little code as possible, this strategy is extremely well-suited for small teams. 

1. Express data structures that will be involved in intensive computation in terms of Numpy arrays. (Because all existing scientific computing Python packages are optimized for those). When designing your data structures, remember that processors are made to processb **contiguous** data. Start with a toy problem, small, and if possible one with an know solution (e.g. analytical)

3. Express your algorithms in terms of operations on those Numpy arrays using as much functionality from you can. Even when this is sub-optimal in runtime. Start developing a simple algorithm 
    - as simple as possible (even if you now that its big-O complexity is not optimal)
    - with as little code as possible,
    - using functionality from existing packages where possible
    - that yields the correct solution (this is essential)

   This can serve as a reference for validating the correctness of later algorithms that are more efficient, but typically also more complex.

4. Profile your code and verify the big-O complexity of your algorithms.

5. Improve the hot-spots:
    - make sure that your algorithms use your data structures in contiguously
    - look for (existing) algorithms with lower big-O functionality.
    - in your own Python code (not in the external functionality called)
        - investigate whether time consuming Python loops can be improved with [Numba](numba.pydata.org)
        - consider reimplementing the python in a compiled low-level language (C, C++, Fortran). Actually, this is the core of this *Best Practice Guide*.

6. If the problem you actually need to solve (not the toy problem)
    - takes prohibitively long on a single core, or
    - is prohibitively large for the memory available to a single core, or
    - both,

   think of parallellization:
    - using GPUs:checkout [Numba](numba.pydata.org), [Jax](https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html), for NVidia GPUs checkout [Cuda Python](https://developer.nvidia.com/cuda/python)
    - using OpenMP: checkout [numpy Global Configuration Options](https://numpy.org/devdocs/reference/global_state.html)
    - using MPI: checkout [mpi4py](mpi4py.readthedocs.io), [dask](https://www.dask.org)
    - for parallelizing linear algebra problems checkout [ELPA](https://elpa.mpcdf.mpg.de), [ScaLAPACK](https://www.netlib.org/scalapack/) (ELPA comes with its own Python wrapper, for ScaLAPACK there is [PyScaLAPACK](https://github.com/etijskens/PyScaLAPACK)

   Note, however, that this step typically adds a lot to the complexity of the code. 

7. Repeat the cycle.

The core feature of this strategy for developing research software is to write as little code as possible and use third party functionality written by specialists wherever possible. Python's `import` statement makes using someone else's code extremely easy, in comparison with compiled languages, and a large amount of high quality Python libraries is available for free. You gain a lot following this approach:

- developer productivity in Python is typically a factor 10 higher that with compiled languages,
- less code is less debugging, less testing, less maintenance, more time for research,
- the code written focusses on your problem domain, not on the numerical details which are handled by the third party libraries, and not (or at least as little as possible) on the syntactical details of compiled code.
- less code and well-readable code also pays off when introducing new students and researchers to the code. Moreover, chances are high that they are already familiar with Python.
- by using an interpreted language the application can be easily extended. No recompilation needed. 

> **Anecdotical evidence**
> 
> Some years ago I was asked by users to profile a commercial code because they found it unacceptably slow. It turned out that most of the compute time was spent in calls to some well-optimized HPC libraries and MPI. Hence, optimization had to come from either better algorithms, or better parallellization. It was a Fortran code that could have been entirely written in Python, without significant performance loss, yet gaining the flexibility of Python. (But that would disclose the code to the users, which was obviously unacceptable to the company.



### Conclusion

A lot of code is glue code, and it is better to write it in a high-level language, outsourcing the hard work to specialized Python packages, and writing low-level code only for performance critical parts while wrapping it in a Python binary extension module. The strategy scheduled above is a top-down strategy: starting out in Python, designing a approach describing the problem you want to solve in Python as well as algorithms to come to a solution. Only the compute intensive parts which cannot be efficiently solved in Python are written in C/C++/Fortran. In that sense the term _C/C++/Fortran back-ends_ is perhaps more appropriate. On the other hand, many scientific computing applications start out in C/C++/Fortran, and the advantages of a Python wrapper are appreciated after a lot of low level code has been writte. Fortunately, it is possible to develop Python wrappers as a front-end to the existing low-level code. This constitutes more of a bottom-up approach.

::: {.callout-note}
Many of the advantages of Python hold also for the Julia programming language. However, its community is significantly smaller as it focuses on scientific programming rather that aiming to be general purpose. It is not typically appearing in student's curricula. It is based on a different programming paradigm than the object-oriented Python. On the other hand it claims that can achieve the performance of C/C++/Fortran on its own. As an example of a succesfull Julia project checkout [DFTK.jl](https://docs.dftk.org)
::: 

### Caveats

1. On modern HPC systems you will typically find distributed file systems which are very good at handling a small number of very large files, but handling a large number of very small files can be problematic. Small files are too small to be distributed, and (nearly) simultaneaously accessing lots of small files creates a lot of work for the metadata file server. Exactly that is what happens when starting up a python process: `import` statements typically refer to small files and often `import` lots of other modules, causing the distributed file system to handle a large number of very small file requests. The load is incurred by every MPI process (easily some 1000s) in the job.
There are ways to avoid this, e.g. creating a container with the Python environment that you need. It is unlikely that this issue will eventually be an obstruction that will diminish the popularity of Python for scientific computing.

2. Distributing packages with binary extension modules is a bit harder than pure Python packages.  

## Approaches for C/C++/Fortran backends

We distinghuish between three approaches for pairing Python with low level code:

### Cython

Cython is basically Python with C types. The source code gets translated into optimized C/C++ code and compiled as Python extension modules. This allows for both very fast program execution and tight integration with external C libraries, while keeping up the high programmer productivity for which the Python language is well known.

### The `ctypes` approach to existing shared libraries

Suppose you come across a C/C++/Fortran library that would be really useful if you could make use of it in your Python code, but does not come with a Python wrapper. The `ctypes` library is part of the Python Standard library and constitutes a foreign function library for Python. It provides C compatible data types, and allows calling functions in DLLs or shared libraries. It can be used to wrap these libraries in pure Python. The shared library is used as is. There is no need to recompile it. 

As an example consider ScaLAPACK. It is available as shared library on nearly every HPC system. However, its developers do not provide Python wrappers. `Ctypes` allows you nevertheless to call all ScaLAPACK functions from Python. [Here](https://github.com/etijskens/PyScaLAPACK) is a wrapper that can be used as a starting point of a `PyScaLAPACK` front-end for an existing ScaLAPACK shared library.

### Creating Python binary extension modules created from C++ or Fortran code

Finally, you can create your own Python binary extension modules which you can just import directly from C++ or Fortran code. 

In the case of Fortran the most obvious tool to use is `numpy.f2py`m. A very nice feature is its automatic support for Numpy arrays. Fortran subroutines and functions accepting arrays are automatically translated into Python wrappers accepting Numpy arrays. 

For C++, there is [Boost.Python](https://www.boost.org/library/latest/python/)[pybind11], (https://github.com/pybind/pybind11), and [nanobind](https://github.com/wjakob/nanobind). Here, you need to write signatures for the wrappers which are then compiled into Python binary extension modules together with the wrapped functions. This is slightly more involved, but then it allows you to expose C++ classes functions and objects to Python, and vice-versa, using no special tools, but your C++ compiler.

For the sake of completeness we must also mention:
- [PyO3](https://pyo3.rs/) which allows to use code written in Rust from Python. 
- [PyJulia](https://github.com/JuliaPy/pyjulia) and [PyCall](https://github.com/JuliaPy/PyCall.jl) can be use to use code written in Julia from Python. See [Incorporating Julia Into Python Programs](https://www.peterbaumgartner.com/blog/incorporating-julia-into-python-programs/)

## The Cython approach

### Pro and contra of `Cython`

## The `ctypes` approach

### Pro and contra of `ctypes`

## Creating Python extension modules from Modern Fortran with `Numpy.f2py`

### Pro and Contra of `f2py`

- Object oriented features of Modern Fortran not supported

## Creating Python extension modules from C++ with `Boost.Python`, `pybind11`, `nanobind`

### Pro and contra of `Boost.Python`, `pybind11`, `nanobind`


## Interesting Links

[Speeding up NumPy with parallelism](https://pythonspeed.com/articles/numpy-parallelism/?utm_source=www.pythonweekly.com&utm_medium=newsletter&utm_campaign=python-weekly-issue-731-february-5-2026&_bhlid=34ea7509a8e22f501d012a734278e24e9d2d675d)

## References {.unnumbered}



:::{#refs}

:::